# NeMo Retriever — Quickstart (deployed service REST API)

This notebook drives a **running** NeMo Retriever deployment (the core stack installed by
`deploy/brev/bootstrap.sh`) entirely over its HTTP API. You will:

1. Health-check the service
2. **Ingest** a multimodal PDF (text + tables + charts) → embeddings → LanceDB
3. Inspect the extracted content
4. **Query** the indexed content (semantic retrieval)
5. (Optional) Generate a grounded **answer** with an LLM

All calls hit the retriever service on port **7670**. Nothing runs in-process here — the
GPU work (page-elements, table-structure, OCR, VL embedding) happens in the NIM pods, and
the index lives in the LanceDB pod.

### Before you start
Port-forward the service from the Brev node (in a terminal):
```bash
kubectl port-forward -n retriever svc/retriever-nemo-retriever 7670:7670
```

In [ ]:
%pip install -q requests
import os, json, time, requests

# Base URL of the deployed retriever service (via port-forward or Ingress).
RETRIEVER_URL = os.environ.get("RETRIEVER_URL", "http://localhost:7670")
print("Retriever service:", RETRIEVER_URL)

## 1. Health check
Confirms the service is up. The core NIMs may still be reported as warming up on a fresh
cluster while model weights finish downloading — re-run until healthy.

In [ ]:
r = requests.get(f"{RETRIEVER_URL}/v1/health", timeout=30)
print(r.status_code)
print(json.dumps(r.json(), indent=2))

## 2. Ingest a document
The ingest API is a **job** you create, then upload one or more documents to. Each upload is
processed asynchronously through the pipeline (extract → embed → write to LanceDB).

We use the repo's `data/multimodal_test.pdf`, which contains text, tables, and a chart.

In [ ]:
from pathlib import Path

# Adjust if you run the notebook from a different working directory.
PDF_PATH = Path(os.environ.get("PDF_PATH", "../../../data/multimodal_test.pdf")).resolve()
assert PDF_PATH.is_file(), f"PDF not found at {PDF_PATH} — set PDF_PATH env var to a valid file."
print("Ingesting:", PDF_PATH)

In [ ]:
# 2a. Create an ingest job. retain_results=True lets us fetch extracted content back.
job_resp = requests.post(
    f"{RETRIEVER_URL}/v1/ingest/job",
    json={
        "expected_documents": 1,
        "label": "quickstart",
        "metadata": {},
        "retain_results": True,
    },
    timeout=60,
)
job_resp.raise_for_status()
job = job_resp.json()
job_id = job["job_id"]
print("job_id:", job_id, "| status:", job.get("status"))

In [ ]:
# 2b. Upload the document to the job (multipart: `file` + a JSON `metadata` field).
metadata = {
    "filename": PDF_PATH.name,
    "content_type": "application/pdf",
    "metadata": {},
    # Optionally pin pipeline behavior, e.g.:
    # "pipeline": {"extract": {"extract_text": True, "extract_tables": True, "extract_charts": True}},
}
with open(PDF_PATH, "rb") as fh:
    up = requests.post(
        f"{RETRIEVER_URL}/v1/ingest/job/{job_id}/document",
        files={"file": (PDF_PATH.name, fh, "application/pdf")},
        data={"metadata": json.dumps(metadata)},
        timeout=120,
    )
up.raise_for_status()
upload = up.json()
document_id = upload["document_id"]
print("document_id:", document_id)

In [ ]:
# 2c. Poll until the document reaches a terminal state (completed / failed).
TERMINAL = {"completed", "failed"}
deadline = time.monotonic() + 600  # 10 min
status = None
while time.monotonic() < deadline:
    docs = requests.get(
        f"{RETRIEVER_URL}/v1/ingest/job/{job_id}/documents",
        params={"limit": 100}, timeout=60,
    ).json()
    items = {d["document_id"]: d for d in docs.get("items", [])}
    rec = items.get(document_id, {})
    status = rec.get("status")
    print(f"status={status} rows={rec.get('result_rows')}", end="\r")
    if status in TERMINAL:
        break
    time.sleep(3)
print("\nfinal status:", status)
if status == "failed":
    print("error:", rec.get("error"))

## 3. Inspect the extracted content
Because we set `retain_results=True`, we can fetch the per-document result payload and see
the extracted text/table/chart chunks the pipeline produced.

In [ ]:
detail = requests.get(
    f"{RETRIEVER_URL}/v1/ingest/job/{job_id}/document/{document_id}", timeout=60
).json()
rows = detail.get("result_data") or []
print(f"extracted {len(rows)} chunk(s)\n")
for i, row in enumerate(rows[:5]):
    # Result rows follow the extraction JSON schema; text lives under metadata.
    meta = row.get("metadata", row)
    text = meta.get("content") or meta.get("text") or json.dumps(row)[:400]
    print(f"--- chunk {i} ---\n{str(text)[:400]}\n")

## 4. Query the indexed content
`/v1/query` embeds your query text (via the VL embed NIM) and does a vector search over
LanceDB, returning the top-k hits. This is the retrieval half of RAG.

In [ ]:
query = "Which animal is jumping onto a laptop?"
q = requests.post(
    f"{RETRIEVER_URL}/v1/query",
    json={"query": query, "top_k": 5, "format": "hits"},
    timeout=120,
)
q.raise_for_status()
results = q.json().get("results", [])
hits = results[0].get("hits", []) if results else []
print(f"{len(hits)} hits for: {query!r}\n")
for h in hits:
    text = h.get("text") or (h.get("metadata", {}) or {}).get("content", "")
    print(f"score={h.get('score')}\n{str(text)[:300]}\n")

## 5. (Optional) Generate a grounded answer

**Option A — external LLM (default for the core-only deployment).** No in-cluster LLM GPU is
used; you pass the retrieved chunks to a hosted model on build.nvidia.com. Requires
`NVIDIA_API_KEY`.

In [ ]:
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")
if NVIDIA_API_KEY and hits:
    context = "\n\n".join(
        str(h.get("text") or (h.get("metadata", {}) or {}).get("content", "")) for h in hits
    )
    prompt = f"Answer the question using only the documents below.\n\nQuestion: {query}\n\nDocuments:\n{context}"
    resp = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {NVIDIA_API_KEY}"},
        json={
            "model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=120,
    )
    resp.raise_for_status()
    print(resp.json()["choices"][0]["message"]["content"])
else:
    print("Set NVIDIA_API_KEY (and run the query cell) to generate an answer via a hosted LLM.")

**Option B — in-cluster answer LLM.** If you deployed the stack with
`--set nimOperator.answer_llm.enabled=true` (adds a ~2-GPU Super-49B NIM), the service
exposes `POST /v1/answer`, which retrieves context and generates in one call. It returns 501
if the LLM is not enabled.

In [ ]:
a = requests.post(
    f"{RETRIEVER_URL}/v1/answer",
    json={"query": query, "top_k": 5, "include_chunks": False},
    timeout=180,
)
if a.status_code == 501:
    print("/v1/answer disabled — no in-cluster LLM. Use Option A instead.")
else:
    a.raise_for_status()
    print(json.dumps(a.json(), indent=2)[:2000])

## 6. Scale up — ingest a small multi-document corpus

The single-PDF flow above is exactly what runs at scale. Here we ingest several
documents in **one job**, watch per-document status, and query across all of them —
each hit shows which source document it came from.

In [ ]:
from pathlib import Path
import time

DATA = Path(os.environ.get("DATA_DIR", "../../../data")).resolve()
corpus = ["multimodal_test.pdf", "table_test.pdf", "test-shapes.pdf",
          "woods_frost.pdf", "functional_validation.pdf", "embedded_table.pdf"]
paths = [DATA / f for f in corpus if (DATA / f).is_file()]
print(f"ingesting {len(paths)} documents from {DATA}")

job = requests.post(f"{RETRIEVER_URL}/v1/ingest/job",
    json={"expected_documents": len(paths), "label": "corpus", "metadata": {}, "retain_results": False},
    timeout=60).json()
jid = job["job_id"]
ids = []
for p in paths:
    meta = {"filename": p.name, "content_type": "application/pdf", "metadata": {}}
    with open(p, "rb") as fh:
        up = requests.post(f"{RETRIEVER_URL}/v1/ingest/job/{jid}/document",
            files={"file": (p.name, fh, "application/pdf")},
            data={"metadata": json.dumps(meta)}, timeout=120).json()
    ids.append(up["document_id"]); print("uploaded", p.name)

Poll the whole job until every document reaches a terminal state, then summarize:

In [ ]:
deadline = time.monotonic() + 1200
items = {}
while time.monotonic() < deadline:
    docs = requests.get(f"{RETRIEVER_URL}/v1/ingest/job/{jid}/documents", params={"limit": 1000}, timeout=60).json()
    items = {d["document_id"]: d for d in docs.get("items", [])}
    done = sum(items.get(i, {}).get("status") in ("completed", "failed") for i in ids)
    print(f"{done}/{len(ids)} documents done", end="\r")
    if done == len(ids): break
    time.sleep(3)
print()
total = 0
for i in ids:
    r = items.get(i, {})
    total += r.get("result_rows") or 0
    print(f"  {str(r.get('status')):9} {str(r.get('result_rows')):>4} chunks  {r.get('filename')}")
print(f"\ntotal chunks indexed across the corpus: {total}")

Now query across the whole corpus — note how hits come from different source documents:

In [ ]:
for query in [
    "What activities are the animals doing?",
    "Show me figures from a table",
    "the road not taken",
]:
    q = requests.post(f"{RETRIEVER_URL}/v1/query",
        json={"query": query, "top_k": 3, "format": "hits"}, timeout=120).json()
    hits = (q.get("results") or [{}])[0].get("hits", [])
    print(f"\nQ: {query}  ->  {len(hits)} hits")
    for h in hits:
        src = (h.get("metadata", {}) or {}).get("source") or h.get("source") or "?"
        txt = h.get("text") or (h.get("metadata", {}) or {}).get("content", "")
        print(f"  [{src}] {str(txt)[:90]}")